# 20.5 照片描述怎樣才算說中了畫面？


一張照片裡有一個人和一輛腳踏車，助理說「有人和腳踏車」，物件都對了，卻還沒有回答「他在做什麼」。同樣兩種物件可以是騎車，也可以是站在車旁。自然圖片的驗收要跟著問題核對，而不是看到回答出現幾個正確名詞，就宣稱整張圖已理解。

前置是[11.4同圖不同問題](https://birdhackor.github.io/tiny-perceptron-vlm/11.4.html)、[11.8換圖後按新真值核對](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)與[11.11分項成績](https://birdhackor.github.io/tiny-perceptron-vlm/11.11.html)。物件是畫面中的東西，動作是可見的活動，關係是東西之間的位置或連結。例如人位於腳踏車旁和人騎在車上，有相同物件，關係與活動卻不同。標註應選圖像能支持的內容，避免替照片補上看不見的目的。

下圖是兩個人工場景示意，並非正式測試照片。上下兩幅都有人和車；上方人物坐在車上騎車，下方人物站在車旁。它用來提醒「物件清單相同」仍可能需要不同回答。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_photo_evidence.svg")))

先用已寫好的兩張回答卡觀察評分差別：


In [ ]:
truth = {"objects": {"人", "腳踏車"}, "activity": "騎車"}
cards = [
    {"objects": {"人", "腳踏車"}, "activity": "騎車"},
    {"objects": {"人", "腳踏車"}, "activity": "站在車旁"},
]
for index, card in enumerate(cards):
    print("卡", index, "物件集合吻合", card["objects"] == truth["objects"])
    print("活動符合本題標註", card["activity"] == truth["activity"])

`truth`是這個人工例子的正確標註，不是從圖自動推得。`objects`用集合，故順序不影響同一組名詞的比較。兩張卡的第一項都應為True，活動則依序True、False。這段沒讀照片或生成描述，只顯示物件與活動不能共用一個判斷。正式自然語言回答有多種合理措辭，不能一律要求整句與某一份描述逐字相同；判分規則要在看答案前訂好，保存實際回答供回查。

完整評估先讓底座回答，再讓本課微調版回答同一份留出照片題；物件、活動和關係各留分母。若描述提到照片上沒有的狗，應保留那項錯誤，不能因為也提對腳踏車就算全對。只問幾道窄題、只看少量照片，也不足以替所有自然場景保證能力。

換圖的檢查仍有用：保持「人正在做什麼」不變，換另一張活動不同的照片，預期答案也隨新照片改變。實際換的是圖片資料，不是把「他正在騎車」偷偷放進文字問題。需要用同一份對話和原始媒體核對，才能區分有用到圖、只依問題猜，以及圖像改變後胡亂換說法。

驗證先提醒我們：兩把評分尺可能給出不同方向。初版微調的整段照片回答通過19/36，底座是11/36；但只看24道短事實題的詞組命中，初版是15/24，底座是21/24。詞組檢查可能漏掉意思正確的另一種說法，也可能接受「先答對物件、後面卻多添錯誤」的句子。初版另有五道整體描述反覆生成，直到長度上限；它們仍留在失敗分母中。[完整驗證回答與逐題判定](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/semantic-validation/paired-384-semantic-review.json)保留了這些差別。

為了檢查更新幅度是否造成問題，我們另從同一底座重跑180步，只把學習率由 `1e-4` 降為 `3e-5`，再用相同驗證題與384個新 token 的上限比較。依事先寫好的退步限制與六類任務成功率平均，較輕更新沒有超過初版，因此在最後測試前固定初版。[選版依據](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/selection.json)與[較輕更新的完整審查](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/semantic-validation/gentle-384-semantic-review.json)保存詳細對照；選到一版，只表示它符合這次教學選版規則，不表示重複與其他錯誤已消失。

最後測試另用12張未拿來訓練或選版的 DOCCI 照片，每張一題整體描述、兩題短事實，共36題；36題來自12張圖，不能當成36張獨立照片。底座與已選定微調版仍共用384個新 token 的上限。完整通過要核對整段：所問物件、活動或關係正確，新增敘述也有圖片支持，而且回答正常結束；有結束符號不代表內容正確，被截斷則不算完成。[原始輸出](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/final/generations-adapter.json)保留完整回答與停止原因。照片由 Jason Baldridge 與家人拍攝，英文描述來自 DOCCI 人工標註者，採 CC BY 4.0；本課使用發布者縮圖並另編中文問題，詳見[資料來源](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/DATA.md)。

[獨立完整回答審查](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/semantic-final/final-semantic-review.json)得到下表。每題按主要要求只歸一欄；數字是**整段語意通過且正常完成的題數／該類全部題數**。

| 本題主要核對什麼 | 底座 | 已選定微調版 |
| --- | ---: | ---: |
| 物件與可見屬性 | 8/13 | 10/13 |
| 活動 | 0/2 | 1/2 |
| 位置或互動關係 | 7/9 | 6/9 |
| 整體場景描述 | 1/12 | 0/12 |
| **全部照片題** | **16/36** | **17/36** |

微調版會答對部分物件與活動短題，全部照片題比底座多通過一題，但關係題與整體描述也有退步。整體描述0/12表示這12份完整回答都沒有同時滿足上述條件，並不表示模型沒有辨認出任何物件。五份描述反覆生成到上限，其餘七份雖正常結束，仍有錯誤、遺漏或圖片不支持的主張。最後測試的短事實詞組命中另為底座23/24、微調版21/24；它們仍不能代替這張完整回答表。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_actual_bike.svg")))

第一個例子是[前輪抬起的機車照片](https://storage.googleapis.com/docci/thumbnails/test_00729.jpg?generation=1713028322603329)。問「摩托車的前輪是在地上，還是抬起來？」時，微調版完整回答「根據圖片中的視角，摩托車的前輪是抬起來的。」，這題通過。但同圖的整體描述又添出「騎士的右腳踩在機車的前輪上，左腳踩在機車的後輪上」。原圖可見一隻腳接地，並沒有雙腳各踩一個輪子的關係，因此整段描述不通過。答對一個窄問題，不能替另一段新增錯誤的描述加分；兩題都正常結束，也不會讓錯誤消失。`test_00729/fact-1` 與 `scene` 的[兩版完整輸出](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/final/generations.json)可一起核對。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_actual_glass.svg")))

第二個例子是[玻璃物件伸到發亮爐口的照片](https://storage.googleapis.com/docci/thumbnails/test_01103.jpg?generation=1713028324564551)。問「人正在如何處理長桿前端的玻璃物件？」時，微調版完整回答「從圖中可以看到，一隻手正握住一個長桿前端的玻璃物件。」照片中的手握著桿柄；回答既把接觸對象說錯，也漏掉把玻璃送進爐口加熱的活動。另問「長桿前端的物件被送到哪裡？」，它答「圖中長桿前端的物件是玻璃吹製機，它被用來將玻璃吹製成圓形。」，仍沒有回答位置。這兩題都不通過：列出手、長桿與玻璃幾個名詞，還不足以回答正在做什麼、東西到哪裡。參考答案稱「玻璃球」只是概括，評分不要求完美球體；照片也看不到吹氣動作，不能把工藝名稱當作畫面已顯示的動作。`test_01103/fact-1`、`fact-2` 的原答案及這些[標註限制](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/semantic-final/final-semantic-review.json)都保留供回查。

另外，我們把同一問題實際送入已公開的選定微調版：「照片裡最主要的東西是什麼？請用一句繁體中文回答，只說看得見的內容。」保持空的先前對話與預設系統設定相同，只更換兩張 TRAIN 圖片。蘑菇照片得到「一個藍色的蘑菇。」，鐵路車廂照片得到「一列鐵路車廂。」；兩份都是完整原回答，正常結束，沒有把物件答案放進輸入。原圖中的蘑菇有可見藍灰色，另一張則確有車廂，主要物件回答隨照片改變。[事前固定的條件](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/image-control.json)與[CPU實際換圖結果](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/image-control/result.json)保存了兩張原圖身分和回答。這只是兩張訓練圖在這組輸入條件下的示範，不能當作盲測成績、完整場景可靠性的證明，靜態車廂照片也不證明車正在行駛。

練習只把第二張卡的`objects`增加「狗」，保持活動不變。先預測物件比較也變False，再執行核對。這個檢查指出回答多添一個可核對的事實，即使文字更豐富，也可能降低照片描述的正確性。
